# G2 — the ExpRAG four-row table (P2.11, C4)

This is C4: the reproduction of ExpRAG, transferred to fact-checking. The table is a 2x2
of **what the student was trained on** against **whether it sees retrieved experience at
inference**.

| row | adapter | retrieval at inference | what it isolates |
|---|---|---|---|
| prompted 3B | none | off | the student with no help at all |
| prompted 3B + retrieval | none | **on** | retrieval alone, no training |
| LoRA SFT | `lora-sft-gold` | off | training alone, no retrieval |
| retrieval-augmented SFT | `rag-sft-gold` | **on** | ExpRAG's actual claim |

ExpRAG's claim is that the fourth row beats the third by more than the second beats the
first — that training *with* retrieved trajectories in the context teaches the model to
use them, over and above retrieval being available. The second row is what makes that
claim falsifiable, and it is the row most easily left out.

**Every row travels the same serving path.** The prompted rows are the same 4-bit
quantisation, chat template and greedy decoding as the adapters, served through
`training.verifier` with `--verifier-base`. Routing the base model through Ollama instead
would have let quantisation and templating differ silently alongside the weights.

**The teacher still rewrites and decomposes** in all four rows. Only the final verdict
comes from the student, because that is the only task it was fine-tuned on. Those teacher
calls are cached from earlier runs and replay from disk.

A fifth and sixth row compare the **filtered** adapters against the gold-target ones, to
check whether fixing the class imbalance actually helped.

In [ ]:
import os
import pathlib
import subprocess
import sys

os.chdir("/kaggle/working")
REPO = pathlib.Path("/kaggle/working/VeriMem")
if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/TERABYTE05/VeriMem.git", str(REPO)],
        check=True,
    )
os.chdir(REPO)
sys.path.insert(0, str(REPO))

RESULTS = "/kaggle/working/results"
os.environ["VERIMEM_DATA_DIR"] = "/kaggle/working/data"
os.environ["VERIMEM_CACHE_DIR"] = "/kaggle/working/cache"
os.environ["VERIMEM_RESULTS_DIR"] = RESULTS
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

LIMIT = "300"  # dev claims per row; 6 rows x 300 student generations

## Inputs and the warm cache

In [ ]:
import pathlib
import shutil

DATA = pathlib.Path("/kaggle/working/data/averitec")
TRAJ = pathlib.Path("/kaggle/working/data/trajectories")
for d in (DATA, TRAJ):
    d.mkdir(parents=True, exist_ok=True)

print("attached:", [p.name for p in pathlib.Path("/kaggle/input").iterdir()])

for name, dest in (
    ("dev.json", DATA),
    ("pool_dev.jsonl", DATA),
    ("trajectories.jsonl", TRAJ),
):
    if (dest / name).exists():
        continue
    for found in pathlib.Path("/kaggle/input").rglob(name):
        shutil.copy(found, dest / name)
        break
    else:
        raise SystemExit(f"{name} not found among the attached inputs")

restored = 0
for ns in ("llm", "evaluator"):
    dst = pathlib.Path(f"/kaggle/working/cache/{ns}")
    if dst.exists():
        continue
    for src in pathlib.Path("/kaggle/input").rglob(f"cache/{ns}"):
        if src.is_dir():
            shutil.copytree(src, dst)
            n = len(list(dst.rglob("*.json")))
            restored += n
            print(f"restored {n} cached {ns} calls")
            break
print(
    f"{restored} cached calls restored"
    if restored
    else "WARNING: cold cache; the teacher calls will all be recomputed"
)

## Find the adapters

Searches `/kaggle/input` **and** `/kaggle/working`, and names what it found. A missing
gold adapter must stop the run rather than quietly produce a four-row table built from
the filtered adapters — the two were trained on different label distributions and the
numbers are not interchangeable.

In [ ]:
ADAPTERS = {}
for root in (pathlib.Path("/kaggle/working"), pathlib.Path("/kaggle/input")):
    if not root.exists():
        continue
    for cfg in root.rglob("adapter/adapter_config.json"):
        # results/<date>_<name>/adapter/ -> the run name identifies the adapter
        run = cfg.parent.parent.name
        key = run.split("_", 1)[-1] if "_" in run else run
        ADAPTERS.setdefault(key, cfg.parent)

for k in sorted(ADAPTERS):
    size = sum(f.stat().st_size for f in ADAPTERS[k].rglob("*") if f.is_file())
    print(f"  {k:<18} {size / 1e6:7.1f} MB  {ADAPTERS[k]}")

NEEDED = ("lora-sft-gold", "rag-sft-gold")
missing = [k for k in NEEDED if k not in ADAPTERS]
if missing:
    raise SystemExit(
        f"missing adapters {missing}. The gold-target runs were trained on a different "
        "label distribution from the filtered ones, so substituting them would report a "
        "table that was never run. Attach the verimem-sft-gold kernel output."
    )
OPTIONAL = [k for k in ("lora-sft", "rag-sft") if k in ADAPTERS]
print(f"\nrequired: {list(NEEDED)}")
print(f"filtered comparison: {OPTIONAL or 'not attached - those two rows will be skipped'}")

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes openai pyyaml python-dotenv 2>&1 | tail -2
import torch

print("cuda:", torch.cuda.is_available(), "| bf16:", torch.cuda.is_bf16_supported())

## The evaluator and the teacher endpoint

In [ ]:
def find_evaluator():
    for root in (pathlib.Path("/kaggle/working"), pathlib.Path("/kaggle/input")):
        if not root.exists():
            continue
        for p in root.rglob("model/config.json"):
            if (p.parent / "model.safetensors").exists() or (
                p.parent / "pytorch_model.bin"
            ).exists():
                return p.parent
    return None


EVALUATOR = find_evaluator()
if EVALUATOR is None:
    raise SystemExit("no evaluator attached - attach the G1 kernel output")
print("EVALUATOR:", EVALUATOR)

In [ ]:
# Ollama queues requests internally at its default, so client-side workers alone
# would buy nothing: the parallelism has to be enabled on both ends. 8/1 means eight
# concurrent requests against one loaded copy of the model -- a second copy would not
# fit in 16 GB beside the evaluator.
os.environ["OLLAMA_NUM_PARALLEL"] = "8"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
os.environ["VERIMEM_WORKERS"] = "8"

import shutil
import subprocess
import time
import urllib.request

if shutil.which("zstd") is None:
    subprocess.run(
        "apt-get -qq update && apt-get -qq install -y zstd", shell=True, capture_output=True
    )
if shutil.which("ollama") is None:
    subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True
    )
OLLAMA_BIN = shutil.which("ollama")
assert OLLAMA_BIN, "ollama did not install"

try:
    urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
except Exception:
    subprocess.Popen(
        [OLLAMA_BIN, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    for _ in range(120):
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("ollama did not start")
subprocess.run([OLLAMA_BIN, "pull", "qwen2.5:7b"], check=True)
print("teacher up", flush=True)

os.environ["VERIMEM_API_BASE_URL"] = "http://localhost:11434/v1"
os.environ["VERIMEM_API_MODEL"] = "qwen2.5:7b"
os.environ.setdefault("VERIMEM_API_KEY", "ollama")

COMMON = [
    "--data",
    "/kaggle/working/data/averitec/dev.json",
    "--store",
    "/kaggle/working/data/averitec/pool_dev.jsonl",
    "--evaluator",
    str(EVALUATOR),
    "--system",
    "crag",
    "--limit",
    LIMIT,
    "--yes",
    "--workers", "8",
]
TRAJ_ARGS = ["--trajectories", "/kaggle/working/data/trajectories/trajectories.jsonl"]

## The four rows

One subprocess per row. Each differs from its neighbour in exactly one thing — the
adapter, or the retrieval flag — so a difference between two rows is attributable to
that one change.

In [ ]:
ROWS = [
    # (name, adapter key or None, retrieval at inference)
    ("g2-prompted-3b", None, False),
    ("g2-prompted-3b-retrieval", None, True),
    ("g2-lora-sft", "lora-sft-gold", False),
    ("g2-rag-sft", "rag-sft-gold", True),
]

# The filtered-vs-gold comparison, only if those adapters are attached.
for key in OPTIONAL:
    ROWS.append((f"g2-{key}-filtered", key, key.startswith("rag")))

for name, key, retrieval in ROWS:
    argv = [sys.executable, "-u", "-m", "eval.run_baselines", *COMMON, "--name", name]
    argv += ["--verifier-base"] if key is None else ["--verifier-adapter", str(ADAPTERS[key])]
    if retrieval:
        argv += [*TRAJ_ARGS, "--experience-retrieval"]
    print(
        f"\n########## {name}  adapter={key or 'none'}  retrieval={retrieval} ##########",
        flush=True,
    )
    r = subprocess.run(argv, env=os.environ)
    if r.returncode != 0:
        raise SystemExit(f"{name} failed")

## The table

`macro-F1` is the headline: AVeriTeC dev is ~61% `Refuted`, so accuracy flatters any
system. The per-class column for `Conflicting evidence` is the one to watch — it is the
class the filtered training data had five examples of, and the reason the gold-target
rerun exists.

In [ ]:
import json
import pathlib

LABELS = ["Supported", "Refuted", "Not enough evidence", "Conflicting evidence"]
found = {}
for f in sorted(pathlib.Path(RESULTS).rglob("metrics.json")):
    if f.parent.name.split("_", 1)[-1].startswith("g2-"):
        found[f.parent.name.split("_", 1)[-1]] = json.loads(f.read_text())

print(
    f"{'row':<26}{'macro-F1':>10}{'accuracy':>10}"
    + "".join(f"{label[:12]:>14}" for label in LABELS)
)
print("-" * (46 + 14 * len(LABELS)))
for name, _, _ in ROWS:
    d = found.get(name)
    if not d:
        print(f"{name:<26}{'(missing)':>10}")
        continue
    # run_baselines nests per-system metrics under the system name.
    m = d.get("crag", d)
    per = m.get("per_class", {})
    line = f"{name:<26}{m.get('macro_f1', 0):>10.3f}{m.get('accuracy', 0):>10.3f}"
    for label in LABELS:
        v = per.get(label, {})
        f1 = v.get("f1") if isinstance(v, dict) else v
        line += f"{'-' if f1 is None else format(f1, '.3f'):>14}"
    print(line)

print("\nExpRAG's claim, stated as a comparison:")
try:
    g = {
        k: found[k].get("crag", found[k])["macro_f1"]
        for k in ("g2-prompted-3b", "g2-prompted-3b-retrieval", "g2-lora-sft", "g2-rag-sft")
    }
    retrieval_alone = g["g2-prompted-3b-retrieval"] - g["g2-prompted-3b"]
    training_alone = g["g2-lora-sft"] - g["g2-prompted-3b"]
    both = g["g2-rag-sft"] - g["g2-prompted-3b"]
    print(f"  retrieval alone        {retrieval_alone:+.3f}")
    print(f"  training alone         {training_alone:+.3f}")
    print(f"  retrieval-aug SFT      {both:+.3f}")
    print(
        f"  super-additive?        {both - retrieval_alone - training_alone:+.3f}"
        "   (> 0 supports ExpRAG)"
    )
except KeyError as exc:
    print(f"  incomplete: {exc}")

In [ ]:
import shutil

shutil.rmtree("/kaggle/working/VeriMem/.git", ignore_errors=True)
!du -sh /kaggle/working/cache /kaggle/working/results